In [ ]:
using Plots
using LinearAlgebra
using Random
using StatsBase
using Statistics
using DelimitedFiles
using JLD2
using CodecZlib
#using ColorSchemes

include("../../../Model/OG/functions/functions_frustration_index_OG.jl")
include("../../../Model/OG/functions/functions_ising_model_finalsweeps_gamma_flipequ_OG.jl")

In [ ]:
base_path = "/data/scratch/projects/punim0638/woodward/nonrcep_casestudy/gamma_data_V3"

function make_filename(base_path, W_name, β_name, σ_name, n_sims, max_sweeps)
    return joinpath(base_path,
        "$(W_name)_$(β_name)_$(σ_name)_$(n_sims)_max$(max_sweeps)sweeps_flipequ.jld2")
end

In [ ]:
function load_gamma_grn_results(
    base_path,
    gammas,
    gamma_names,
    grns;
    W_range,
    W_sign = 1,
    β_name,
    σ_name,
    n_sims,
    max_sweeps,
    names = [
        "sigma_finals_selfish",
        "E_selfish",
        "avg_flip_counts",
        "avg_tissue_frustration",
        "sweeps_used",
    ]
)
    # empty containers: results[K_name][γ][name] = one entry per W
    results = Dict()
    for K_name in grns
        results[K_name] = Dict()
        for γ in gammas
            results[K_name][γ] = Dict(name => [] for name in names)
        end
    end

    W_loaded = Int[]

    for i in W_range
        W_name = W_sign == 1 ? "W$(i)" : "-W$(i)"
        filename = make_filename(base_path, W_name, β_name, σ_name, n_sims, max_sweeps)

        if !isfile(filename)
            @warn "Missing file for $(W_name), skipping"
            continue
        end

        jldopen(filename, "r") do f
            for K_name in grns
                for (γ, γ_name) in zip(gammas, gamma_names)
                    for name in names
                        push!(results[K_name][γ][name], f["$(γ_name)/$(K_name)/$(name)"])
                    end
                end
            end
        end
        push!(W_loaded, i)
    end

    return results, W_loaded
end


In [ ]:
### ----------- Imports ----------- ###
K_tree = Float64.([
    0  1  1  0  0  0  0  0  0;
    0  0  0  1  0  0  0  0  0;
    0  0  0  0  1  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0  0  0  0  0  0  0  0  0;
    0  0  0  0  0  0  0  0  0;
    0  0  0  0  0  0  0  0  0
])

K_negtri = Float64.([
    0   1  0  0  0  0  0  0  0;
    0   0  1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_ICFF = Float64.([
    0   1  -1  0  0  0  0  0  0;
    0   0   1  1  0  0  0  0  0;
    0   0   0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0
])

K_CFF = Float64.([
    0   1   1  0  0  0  0  0  0;
    0   0   1  1  0  0  0  0  0;
    0   0   0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0
])

K_negtritri = Float64.([
    0   -1  0  0  0  0  0  0  0;
    0   0  -1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_negtritwin = Float64.([
    0   -1  0  0  0  0  0  0  0;
    0   0  1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_treesym = Matrix(Symmetric(copy(K_tree), :U))+ Matrix(Symmetric(copy(K_tree), :L))
K_negtrisym = Matrix(Symmetric(copy(K_negtri), :U))+ Matrix(Symmetric(copy(K_negtri), :L))
K_ICFFsym = Matrix(Symmetric(copy(K_ICFF), :U))+ Matrix(Symmetric(copy(K_ICFF), :L))
K_CFFsym = Matrix(Symmetric(copy(K_CFF), :U))+ Matrix(Symmetric(copy(K_CFF), :L));
K_negtritrisym = Matrix(Symmetric(copy(K_negtritri), :U))+ Matrix(Symmetric(copy(K_negtritri), :L));
K_negtritwinsym = Matrix(Symmetric(copy(K_negtritwin), :U))+ Matrix(Symmetric(copy(K_negtritwin), :L));

In [ ]:
### ----------- constants ----------- ###
n_cells = 400
n_genes = 9 

β_test = Inf
n_sims = 10 
max_sweeps = 10_000

β_name = "betaInf"
σ_name = "σrand"
max_sweeps_name = "$(max_sweeps)"

In [ ]:
gammas      = [1.0, 0.8, 0.5, 0.3, 0.25, 0.22, 0.2, 0.18, 0.16, 0.14, 0.125, 0.12, 0.1, 0.08, 0.07, 0.06, 0.05, 0.04, 0.001, 0.0]
gamma_names = ["g1", "g08", "g05", "g03", "g025", "g022", "g02", "g018", "g016", "g014", "g0125", "g012", "g01", "g008", "g007", "g006", "g005", "g004", "g0001", "g0"]

grns        = ["tree", "negtri", "ICFF", "CFF", "negtritwin", "negtritri"]

In [ ]:
results, W_loaded = load_gamma_grn_results(
    base_path, gammas, gamma_names, grns;
    W_range    = 1:100,
    β_name     = β_name,
    σ_name     = σ_name,
    n_sims     = n_sims,
    max_sweeps = max_sweeps
);

In [ ]:
function average_over_Ws(data)
    n_W      = length(data)        # number of Ws loaded
    n_sweeps = length(data[1])     # 10000

    avg = zeros(n_sweeps)

    for s in 1:n_sweeps
        vals = [data[w][s] for w in 1:n_W]     # sweep s from every W
        avg[s] = mean(filter(!isnan, vals))    # average, ignoring NaN
    end

    sweeps = findall(!isnan, avg)
    values = avg[sweeps]

    return sweeps, values
end

In [ ]:
Wavg = Dict()

for K_name in grns
    Wavg[K_name] = Dict()
    for γ in gammas
        sweeps, flips = average_over_Ws(results[K_name][γ]["avg_flip_counts"])
        _,      frus  = average_over_Ws(results[K_name][γ]["avg_tissue_frustration"])

        Wavg[K_name][γ] = Dict(
            "sweeps"                 => sweeps,
            "avg_flip_counts"        => flips,
            "avg_tissue_frustration" => frus
        )
    end
end

# Final avg energy
## averged over the n_sims and over the range of Ws

In [ ]:
final_E_avg = Dict()

for K_name in grns
    final_E_avg[K_name] = Dict()

    for γ in gammas
        E_list  = results[K_name][γ]["E_selfish"]      # one matrix per W (sweeps × sims)
        W_means = zeros(length(E_list))

        for w in 1:length(E_list)
            E = E_list[w]

            finals = zeros(size(E, 2))
            for sim in 1:size(E, 2)
                finals[sim] = filter(!isnan, E[:, sim])[end]   # final lattice energy of this sim
            end

            W_means[w] = mean(finals)        # 1. average over sims
        end

        final_E_avg[K_name][γ] = mean(W_means)   # 2. average over Ws
    end
end


In [ ]:
grn_colors = Dict(
    "tree"   => :orange,
    "tri"    => :green,
    "negtri" => :teal,
    "ICFF"   => :purple,
    "CFF"    => :violet,
    "negtritwin" => :blue,
    "negtritri" => :cyan
)

In [ ]:
x = 1:length(gammas)

p = plot(
    xlabel = "γ",
    ylabel = "Final energy (avg over sims, then Ws)",
    title  = "Final tissue energy vs γ",
    xticks = (x, string.(gammas)),
    xrotation = 45
)

for (i, K_name) in enumerate(grns)
    y = [final_E_avg[K_name][γ] for γ in gammas]
    plot!(p, x, y; marker = :circle, linewidth = 2, label = K_name, color = grn_colors[K_name])
end

display(p)

#savefig(p, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/final_tissue_energy_gammarange_$(σ_name)_set1_V3_V3.png")

# Avg flip counts - should be zero as this is the equlibirum condition

In [ ]:
F = results["negtri"][1.0]["avg_flip_counts"]

println(length(F))               # number of Ws loaded
println(length(F[1]))            # should be 10000
println(count(!isnan, F[1]))     # sweeps with data for the first W (should be ~50–500)
println(filter(!isnan, F[1])[end])   # final value for the first W


In [ ]:
final_flips_avg = Dict()
final_flips_std = Dict()

for K_name in grns
    final_flips_avg[K_name] = Dict()
    final_flips_std[K_name] = Dict()

    for γ in gammas
        W_finals = [filter(!isnan, F)[end] for F in results[K_name][γ]["avg_flip_counts"]]

        final_flips_avg[K_name][γ] = mean(W_finals)
        final_flips_std[K_name][γ] = std(W_finals)
    end
end

In [ ]:
x = 1:length(gammas)

p = plot(
    xlabel    = "γ",
    ylabel    = "Final avg flips per sweep",
    title     = "Final flips vs γ",
    xticks    = (x, string.(gammas)),
    xrotation = 45
)

for K_name in grns
    y = [final_flips_avg[K_name][γ] for γ in gammas]./ n_cells  # flips per cell per sweep

    plot!(p, x, y;
        marker    = :circle,
        linewidth = 2,
        label     = K_name,
        color     = grn_colors[K_name]
    )
end

display(p)
#savefig(p,  "~/my_folder/compute/data/gamma_study/gamma_plots_set1/finaltissue_flips_gammarange_$(σ_name)_set1_V3.png")

# avg tissue frustartion density

In [ ]:
final_frus_avg = Dict()

for K_name in grns
    final_frus_avg[K_name] = Dict()

    for γ in gammas
        W_finals = [filter(!isnan, F)[end] for F in results[K_name][γ]["avg_tissue_frustration"]]

        final_frus_avg[K_name][γ] = mean(W_finals)    # average over Ws
    end
end


In [ ]:
gammas_plot = filter(!=(0.0), gammas)
x = 1:length(gammas_plot)

p = plot(
    xlabel    = "γ",
    ylabel    = "Final tissue frustration density",
    title     = "Final frustration vs γ",
    xticks    = (x, string.(gammas_plot)),
    xrotation = 45
)

for K_name in grns
    y = [final_frus_avg[K_name][γ] for γ in gammas_plot]

    plot!(p, x, y;
        marker    = :circle,
        linewidth = 2,
        label     = K_name,
        color     = grn_colors[K_name]
    )
end

display(p) 
#savefig(p,  "~/my_folder/compute/data/gamma_study/gamma_plots_set1/finaltissue_frustration_gammarange_$(σ_name)_set1_V3.png")